# Mixture-of-Experts language models

Mixture-of-Experts models grow capacity while activating only a few experts per token.

Sparse routing chooses a small set of experts for each token. Gates, top-k normalization, load balancing, and capacity limits determine whether the extra capacity is actually usable. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(96)


def softmax(logits):
    values = np.asarray(logits, dtype=float)
    shifted = values - values.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)


EXPERT_OUTPUTS = np.array([
    [1.0, 0.0],
    [0.0, 2.0],
    [0.3, 0.4],
    [0.7, 0.2],
])


def moe_ladder():
    return [
        {"name": "D1 one prompt", "domains": [0], "logits": [[2.0, 1.0, 0.0, -1.0]], "capacity": 3},
        {"name": "D2 few-shot routing set", "domains": [0, 1, 2, 0], "logits": [[2, 1, 0, -1], [0, 2, 1, -1], [0, 1, 2, -1], [2, 0, 1, -1]], "capacity": 3},
        {"name": "D3 distractors and skew", "domains": [0, 0, 0, 1, 2, 0], "logits": [[3, 1, 0, -1], [2.5, 1, 0, -1], [2.2, 0, 1, -1], [1.8, 1.7, 0, -1], [2.0, 0, 1.5, -1], [2.7, 1, 0, -1]], "capacity": 2},
        {"name": "D4 task tool set", "domains": [0, 1, 2, 3, 1], "logits": [[2, 1, 0, 0], [0, 2, 1, 0], [0, 1, 2, 0], [0, 1, 0, 2], [0, 2.2, 1, 0]], "capacity": 2},
        {"name": "D5 longer context router collapse", "domains": [0, 1, 2, 3, 1, 2, 0, 3], "logits": [[3, 1, 0, 0], [3, 1.5, 0, 0], [3, 0, 1.5, 0], [3, 0, 0, 1.5], [3, 1.4, 0, 0], [3, 0, 1.2, 0], [3, 1, 0, 0], [3, 0, 0, 1.2]], "capacity": 2},
    ]

## The concept, built once: sparse top-k routing

A Mixture-of-Experts layer routes each token through only selected experts:
$$y=\sum_{e\in \mathrm{TopK}(r(x))}g_e(x)E_e(x).$$
The gates $g_e$ are the softmax weights renormalized over the top-k experts.

In [ ]:
def route_to_experts(router_logits, expert_outputs, top_k=2, capacity=3):
    logits = np.asarray(router_logits, dtype=float)
    gates = softmax(logits)
    top_indices = np.argsort(gates)[-top_k:][::-1]
    top_weights = gates[top_indices]
    renormalized = top_weights / top_weights.sum()
    mixture = renormalized @ expert_outputs[top_indices]
    active_fraction = top_k / 16.0
    loads = np.bincount(top_indices, minlength=len(gates))
    overflow = int(np.maximum(loads - capacity, 0).sum())
    return {
        "gates": gates,
        "top_indices": top_indices,
        "renormalized": renormalized,
        "mixture": mixture,
        "active_fraction": active_fraction,
        "overflow": overflow,
    }

The lesson numbers are softmax $[2,1,0]\to[0.665,0.245,0.090]$, top-2 renormalization $[0.731,0.269]$, active fraction $2/16=12.5\%$, loads $[2,5]$ with capacity 3 drop 2 tokens, and expert outputs mix to $[0.731,0.538]$.

In [ ]:
built = route_to_experts([2.0, 1.0, 0.0], np.array([[1.0, 0.0], [0.0, 2.0], [0.3, 0.4]]), top_k=2, capacity=3)
loads = np.array([2, 5])
overflow = int(np.maximum(loads - 3, 0).sum())

assert np.allclose(np.round(built["gates"], 3), np.array([0.665, 0.245, 0.090]))
assert np.allclose(np.round(built["renormalized"], 3), np.array([0.731, 0.269]))
assert np.allclose(np.round(built["mixture"], 3), np.array([0.731, 0.538]))
assert round(built["active_fraction"] * 100, 1) == 12.5
assert overflow == 2

print("gates", np.round(built["gates"], 3))
print("renormalized top-2", np.round(built["renormalized"], 3))
print("mixture", np.round(built["mixture"], 3))
print("active percent", round(built["active_fraction"] * 100, 1))
print("overflow", overflow)

## The dataset ladder

D1-D5 is an inline prompt/context ladder: a single prompt, few-shot routing, distractors, real-ish tools, and a longer context where the router collapses to one expert.

In [ ]:
ladder = moe_ladder()
for rung in ladder:
    logits = np.asarray(rung["logits"], dtype=float)
    print(rung["name"], "tokens", len(rung["domains"]), "experts", logits.shape[1], "capacity", rung["capacity"])

In [ ]:
rows = []
for rung in ladder:
    assignments = []
    correct = 0
    for logits, domain in zip(rung["logits"], rung["domains"]):
        result = route_to_experts(logits, EXPERT_OUTPUTS, top_k=2, capacity=rung["capacity"])
        assignment = int(result["top_indices"][0])
        assignments.append(assignment)
        correct += int(assignment == domain)
    loads = np.bincount(assignments, minlength=EXPERT_OUTPUTS.shape[0])
    overflow_rate = float(np.maximum(loads - rung["capacity"], 0).sum()) / max(1, len(assignments))
    accuracy = correct / len(assignments)
    rows.append({
        "name": rung["name"],
        "context": len(assignments),
        "accuracy": accuracy,
        "overflow": overflow_rate,
        "loads": loads,
    })
for row in rows:
    print(f"{row['name']}: acc={row['accuracy']:.3f} overflow={row['overflow']:.3f} loads={row['loads'].tolist()}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(rows):
    axes[0, idx].bar(np.arange(len(row["loads"])), row["loads"], color="teal")
    axes[0, idx].set_title(f"D{idx + 1} loads")
    axes[1, idx].bar(["acc", "overflow"], [row["accuracy"], row["overflow"]], color=["teal", "tomato"])
    axes[1, idx].set_ylim(0, 1)
fig.suptitle("Expert-load panels and routing quality")
plt.figure(figsize=(6, 3))
plt.plot([row["context"] for row in rows], [row["accuracy"] for row in rows], marker="o", label="accuracy")
plt.plot([row["context"] for row in rows], [row["overflow"] for row in rows], marker="s", label="overflow")
plt.xlabel("context tokens")
plt.ylabel("routing metric")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.grid(True)

## Pitfall on D5: router collapse

If the router sends most tokens to one expert, sparse capacity becomes a queue. A load-balancing correction subtracts pressure from overloaded experts before top-k selection.

In [ ]:
hard = ladder[-1]
collapsed_assignments = [int(np.argmax(logits)) for logits in hard["logits"]]
collapsed_loads = np.bincount(collapsed_assignments, minlength=4)
collapsed_overflow = int(np.maximum(collapsed_loads - hard["capacity"], 0).sum())
balanced_assignments = []
running_loads = np.zeros(4, dtype=float)
for logits in hard["logits"]:
    adjusted = np.asarray(logits, dtype=float) - 0.8 * running_loads
    assignment = int(np.argmax(adjusted))
    balanced_assignments.append(assignment)
    running_loads[assignment] += 1
balanced_loads = np.bincount(balanced_assignments, minlength=4)
balanced_overflow = int(np.maximum(balanced_loads - hard["capacity"], 0).sum())
print("collapsed loads", collapsed_loads.tolist(), "overflow", collapsed_overflow)
print("balanced loads", balanced_loads.tolist(), "overflow", balanced_overflow)

## Evaluate it + practice

            - Metric: routing accuracy and overflow rate; compare to the no-skill baseline, always route to the largest training domain.
            - Cheap sanity check: top-k gates renormalize to sum to 1.
            - Ablation: remove load balancing and inspect D5 overflow.
            - Failure signals: one expert receives most tokens or overflow silently drops data.
            - Practice: Change top_k from 2 to 1 and recompute active fraction.
- Practice: Raise D5 capacity and plot overflow.
- Practice: Add a domain-specific expert and inspect assignments.